In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

In [3]:
df = pd.read_csv("housing.csv")

In [4]:
df['income_cat'] = pd.cut(df['median_income'], bins = [0, 1.5, 3.0, 4.5, 6.0, 7.5, np.inf], labels = [1, 2, 3 ,4 ,5, 6 ])

In [5]:
split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)
for train_index, test_index in split.split(df, df['income_cat']):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

In [6]:
#Remove income_cat column
for x in (strat_train_set, strat_test_set, df):
    x.drop("income_cat", axis = 1, inplace = True)

In [7]:
strat_train_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
6895,-118.11,34.04,28.0,3913.0,696.0,2264.0,697.0,5.2446,258000.0,<1H OCEAN
2880,-118.97,35.38,35.0,1673.0,426.0,1041.0,413.0,1.3750,57500.0,INLAND
13769,-117.03,34.07,16.0,3784.0,577.0,1615.0,525.0,4.2333,220300.0,INLAND
18349,-122.17,37.43,24.0,3924.0,1142.0,7174.0,950.0,4.0972,387500.0,NEAR OCEAN
16188,-121.31,37.96,48.0,1112.0,227.0,583.0,216.0,2.3393,77600.0,INLAND


In [8]:
df = strat_train_set.copy()

In [9]:
housing = strat_train_set.drop("median_house_value", axis = 1)
housing_labels = strat_train_set["median_house_value"]

In [10]:
from sklearn.impute import SimpleImputer

In [11]:
imputer = SimpleImputer(strategy = "median")

In [12]:
housing_num = housing.select_dtypes(include = [np.number])
imputer.fit(housing_num)

SimpleImputer(strategy='median')

In [13]:
X = imputer.transform(housing_num)#X is transformed into numpy arrsay
housing = pd.DataFrame(X, columns = housing_num.columns, index = housing_num.index)
# To transform X again into Pandas Data 

In [14]:
housing.describe()
#The missing values of toatal_bedrooms is filled

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
count,16512.000000,16512.00000,16512.000000,16512.000000,16512.000000,16512.000000,16512.00000,16512.000000
mean,-119.572346,35.63674,28.628513,2628.114583,534.710695,1422.758600,497.75327,3.873473
std,2.001511,2.13778,12.586026,2157.526589,411.518229,1120.859666,376.32796,1.900464
min,-124.350000,32.54000,1.000000,6.000000,1.000000,3.000000,1.00000,0.499900
25%,-121.800000,33.93000,18.000000,1445.000000,297.000000,786.000000,279.00000,2.566775
50%,-118.510000,34.26000,29.000000,2123.500000,433.500000,1166.000000,408.00000,3.540900
75%,-118.010000,37.72000,37.000000,3143.250000,642.000000,1720.000000,603.00000,4.744475
max,-114.310000,41.95000,52.000000,39320.000000,6210.000000,35682.000000,5358.00000,15.000100


In [15]:
housing['ocean_proximity'] = df['ocean_proximity']

In [16]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
6895,-118.11,34.04,28.0,3913.0,696.0,2264.0,697.0,5.2446,<1H OCEAN
2880,-118.97,35.38,35.0,1673.0,426.0,1041.0,413.0,1.3750,INLAND
13769,-117.03,34.07,16.0,3784.0,577.0,1615.0,525.0,4.2333,INLAND
18349,-122.17,37.43,24.0,3924.0,1142.0,7174.0,950.0,4.0972,NEAR OCEAN
16188,-121.31,37.96,48.0,1112.0,227.0,583.0,216.0,2.3393,INLAND
...,...,...,...,...,...,...,...,...,...
10505,-117.72,33.43,5.0,1889.0,359.0,616.0,246.0,3.8992,<1H OCEAN
19526,-120.97,37.65,16.0,3960.0,716.0,1776.0,724.0,3.9886,INLAND
9347,-122.58,37.98,52.0,1180.0,216.0,467.0,197.0,4.9615,NEAR BAY
3650,-118.44,34.21,37.0,1665.0,335.0,1011.0,343.0,4.8703,<1H OCEAN


In [17]:
house1 = housing.copy()
house2 = housing[['ocean_proximity']]#In One hot encoder we will only work with this columns

In [18]:
from sklearn.preprocessing import OrdinalEncoder

In [19]:
ordinalencoder = OrdinalEncoder()
housing_cat1 = ordinalencoder.fit_transform(house1)

In [20]:
house1 = pd.DataFrame(housing_cat1, columns = house1.columns, index = house1.index) 

In [21]:
house1
# !! here problem is ML can learn that 00 is far from 04 and closer to 01 which can lead to errors

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
6895,574.0,150.0,27.0,3591.0,696.0,2192.0,696.0,8110.0,0.0
2880,488.0,274.0,34.0,1467.0,425.0,1001.0,412.0,369.0,1.0
13769,682.0,153.0,15.0,3484.0,577.0,1575.0,524.0,6489.0,1.0
18349,168.0,461.0,23.0,3598.0,1130.0,3542.0,948.0,6193.0,4.0
16188,254.0,514.0,47.0,909.0,226.0,543.0,215.0,2221.0,1.0
...,...,...,...,...,...,...,...,...,...
10505,613.0,89.0,4.0,1683.0,358.0,576.0,245.0,5791.0,0.0
19526,288.0,483.0,15.0,3623.0,716.0,1735.0,723.0,5943.0,1.0
9347,127.0,516.0,51.0,974.0,215.0,427.0,196.0,7562.0,3.0
3650,541.0,167.0,36.0,1459.0,334.0,971.0,342.0,7464.0,0.0


In [22]:
from sklearn.preprocessing import OneHotEncoder

In [23]:
onehotencoder = OneHotEncoder()
housing_cat2 = onehotencoder.fit_transform(house2)

In [24]:
onehotencoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [25]:
house2 = pd.DataFrame(housing_cat2.toarray(), columns = ['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'], index = house2.index) 

In [26]:
house2

,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
6895,1.0,0.0,0.0,0.0,0.0
2880,0.0,1.0,0.0,0.0,0.0
13769,0.0,1.0,0.0,0.0,0.0
18349,0.0,0.0,0.0,0.0,1.0
16188,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...
10505,1.0,0.0,0.0,0.0,0.0
19526,0.0,1.0,0.0,0.0,0.0
9347,0.0,0.0,0.0,1.0,0.0
3650,1.0,0.0,0.0,0.0,0.0


In [27]:
df = pd.concat([housing, house2], axis = 1)

In [28]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
6895,-118.11,34.04,28.0,3913.0,696.0,2264.0,697.0,5.2446,<1H OCEAN,1.0,0.0,0.0,0.0,0.0
2880,-118.97,35.38,35.0,1673.0,426.0,1041.0,413.0,1.3750,INLAND,0.0,1.0,0.0,0.0,0.0
13769,-117.03,34.07,16.0,3784.0,577.0,1615.0,525.0,4.2333,INLAND,0.0,1.0,0.0,0.0,0.0
18349,-122.17,37.43,24.0,3924.0,1142.0,7174.0,950.0,4.0972,NEAR OCEAN,0.0,0.0,0.0,0.0,1.0
16188,-121.31,37.96,48.0,1112.0,227.0,583.0,216.0,2.3393,INLAND,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10505,-117.72,33.43,5.0,1889.0,359.0,616.0,246.0,3.8992,<1H OCEAN,1.0,0.0,0.0,0.0,0.0
19526,-120.97,37.65,16.0,3960.0,716.0,1776.0,724.0,3.9886,INLAND,0.0,1.0,0.0,0.0,0.0
9347,-122.58,37.98,52.0,1180.0,216.0,467.0,197.0,4.9615,NEAR BAY,0.0,0.0,0.0,1.0,0.0
3650,-118.44,34.21,37.0,1665.0,335.0,1011.0,343.0,4.8703,<1H OCEAN,1.0,0.0,0.0,0.0,0.0


In [29]:
df = df.drop("ocean_proximity", axis = 1)

In [30]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
6895,-118.11,34.04,28.0,3913.0,696.0,2264.0,697.0,5.2446,1.0,0.0,0.0,0.0,0.0
2880,-118.97,35.38,35.0,1673.0,426.0,1041.0,413.0,1.3750,0.0,1.0,0.0,0.0,0.0
13769,-117.03,34.07,16.0,3784.0,577.0,1615.0,525.0,4.2333,0.0,1.0,0.0,0.0,0.0
18349,-122.17,37.43,24.0,3924.0,1142.0,7174.0,950.0,4.0972,0.0,0.0,0.0,0.0,1.0
16188,-121.31,37.96,48.0,1112.0,227.0,583.0,216.0,2.3393,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10505,-117.72,33.43,5.0,1889.0,359.0,616.0,246.0,3.8992,1.0,0.0,0.0,0.0,0.0
19526,-120.97,37.65,16.0,3960.0,716.0,1776.0,724.0,3.9886,0.0,1.0,0.0,0.0,0.0
9347,-122.58,37.98,52.0,1180.0,216.0,467.0,197.0,4.9615,0.0,0.0,0.0,1.0,0.0
3650,-118.44,34.21,37.0,1665.0,335.0,1011.0,343.0,4.8703,1.0,0.0,0.0,0.0,0.0


In [31]:
from sklearn.preprocessing import MinMaxScaler

In [32]:
scaler = MinMaxScaler(feature_range=(-1,1))
df_scaled1 = scaler.fit_transform(df)

In [33]:
df_scaled1 = pd.DataFrame(df_scaled1, columns = df.columns, index = df.index)

In [34]:
df_scaled1

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
6895,0.243028,-0.681190,0.058824,-0.801241,-0.776131,-0.873259,-0.740153,-0.345568,1.0,-1.0,-1.0,-1.0,-1.0
2880,0.071713,-0.396387,0.333333,-0.915196,-0.863102,-0.941815,-0.846183,-0.879298,-1.0,1.0,-1.0,-1.0,-1.0
13769,0.458167,-0.674814,-0.411765,-0.807804,-0.814463,-0.909639,-0.804368,-0.485055,-1.0,1.0,-1.0,-1.0,-1.0
18349,-0.565737,0.039320,-0.098039,-0.800682,-0.632469,-0.598027,-0.645697,-0.503828,-1.0,-1.0,-1.0,-1.0,1.0
16188,-0.394422,0.151966,0.843137,-0.943735,-0.927202,-0.967488,-0.919731,-0.746293,-1.0,1.0,-1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10505,0.320717,-0.810840,-0.843137,-0.904207,-0.884684,-0.965638,-0.908531,-0.531138,1.0,-1.0,-1.0,-1.0,-1.0
19526,-0.326693,0.086079,-0.411765,-0.798850,-0.769689,-0.900614,-0.730073,-0.518807,-1.0,1.0,-1.0,-1.0,-1.0
9347,-0.647410,0.156217,1.000000,-0.940276,-0.930746,-0.973990,-0.926825,-0.384615,-1.0,-1.0,-1.0,1.0,-1.0
3650,0.177291,-0.645058,0.411765,-0.915603,-0.892414,-0.943496,-0.872317,-0.397195,1.0,-1.0,-1.0,-1.0,-1.0


In [35]:
from sklearn.preprocessing import StandardScaler

In [36]:
sd_scaler = StandardScaler()
df_scaled2 = sd_scaler.fit_transform(df)

In [37]:
df_scaled2 = pd.DataFrame(df_scaled2, columns = df.columns, index = df.index)

In [38]:
df_scaled2

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
6895,0.730643,-0.746937,-0.049939,0.595554,0.391949,0.750555,0.529466,0.721492,1.127082,-0.684483,-0.011006,-0.355754,-0.383179
2880,0.300955,-0.120100,0.506250,-0.442703,-0.264178,-0.340605,-0.225218,-1.314705,-0.887247,1.460957,-0.011006,-0.355754,-0.383179
13769,1.270252,-0.732904,-1.003406,0.535762,0.102767,0.171518,0.072404,0.189342,-0.887247,1.460957,-0.011006,-0.355754,-0.383179
18349,-1.297886,0.838867,-0.367761,0.600653,1.475773,5.131254,1.201772,0.117726,-0.887247,-0.684483,-0.011006,-0.355754,2.609748
16188,-0.868198,1.086796,1.539173,-0.702731,-0.747768,-0.749232,-0.748713,-0.807287,-0.887247,1.460957,-0.011006,-0.355754,-0.383179
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10505,0.925502,-1.032289,-1.877418,-0.342585,-0.426994,-0.719790,-0.668993,0.013538,1.127082,-0.684483,-0.011006,-0.355754,-0.383179
19526,-0.698321,0.941781,-1.003406,0.617339,0.440551,0.315162,0.601214,0.060580,-0.887247,1.460957,-0.011006,-0.355754,-0.383179
9347,-1.502738,1.096151,1.856996,-0.671212,-0.774499,-0.852727,-0.799203,0.572524,-0.887247,-0.684483,-0.011006,2.810934,-0.383179
3650,0.565763,-0.667413,0.665162,-0.446411,-0.485317,-0.367371,-0.411232,0.524534,1.127082,-0.684483,-0.011006,-0.355754,-0.383179
